# 02. Preprocessing Baseline

Tiền xử lý dữ liệu Abalone cơ bản:
- Load dữ liệu
- Kiểm tra và xử lý giá trị thiếu
- Kiểm tra và xử lý outliers
- Mã hóa biến phân loại
- Chuẩn hóa/scaling dữ liệu (optional)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Thêm src vào đường dẫn
sys.path.insert(0, str(Path.cwd().parent.parent / 'src'))

from data.load_data import load_abalone_data

# Cấu hình hiển thị
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


## 1. Load Data

In [ ]:
# Đọc dữ liệu từ thư mục raw
data_path = Path.cwd().parent.parent / 'data' / 'raw' / 'abalone.csv'
df = load_abalone_data(data_path)

print(f"Hình dạng dữ liệu gốc: {df.shape}")
print("\nVài hàng đầu tiên:")
print(df.head())
print("\nKiểu dữ liệu:")
print(df.dtypes)
print("\nThống kê cơ bản:")
print(df.describe())


## 2. Kiểm Tra Giá Trị Thiếu & Chất Lượng Dữ Liệu

In [ ]:
# Kiểm tra giá trị thiếu
print("Giá trị thiếu:")
print(df.isnull().sum())
print(f"\nTổng số giá trị thiếu: {df.isnull().sum().sum()}")

# Kiểm tra dòng trùng lặp
print(f"\nDòng trùng lặp: {df.duplicated().sum()}")

# Kiểm tra kiểu dữ liệu
print("\nKiểu dữ liệu:")
print(df.dtypes)

# Kiểm tra giá trị duy nhất trong cột phân loại
print("\nGiá trị duy nhất trong cột 'Sex':")
print(df['Sex'].unique())
print(f"Số lượng: \n{df['Sex'].value_counts()}")


## 3. Xử Lý Biến Phân Loại (Mã Hóa)

In [ ]:
# Tạo bản sao để xử lý
df_processed = df.copy()

# Mã hóa 'Sex' sử dụng phương pháp one-hot encoding
print("Hình dạng trước mã hóa:", df_processed.shape)
df_processed = pd.get_dummies(df_processed, columns=['Sex'], prefix='Sex', drop_first=False)
print("Hình dạng sau mã hóa:", df_processed.shape)

print("\nCác cột sau mã hóa:")
print(df_processed.head())


## 4. Phát Hiện & Xử Lý Giá Trị Bất Thường

In [ ]:
# Phát hiện giá trị bất thường sử dụng phương pháp IQR
def detect_outliers_iqr(data, columns, multiplier=1.5):
    """Phát hiện giá trị bất thường sử dụng phương pháp IQR"""
    outliers_mask = pd.Series([False] * len(data))
    outlier_info = {}
    
    for col in columns:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - multiplier * IQR
        upper_bound = Q3 + multiplier * IQR
        
        col_outliers = (data[col] < lower_bound) | (data[col] > upper_bound)
        outliers_mask = outliers_mask | col_outliers
        
        outlier_info[col] = {
            'lower_bound': lower_bound,
            'upper_bound': upper_bound,
            'count': col_outliers.sum()
        }
    
    return outliers_mask, outlier_info

# Các cột số (loại trừ cột Sex đã mã hóa và cột mục tiêu Rings)
numeric_cols = df_processed.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if not col.startswith('Sex_') and col != 'Rings']

print(f"Các cột số để phát hiện giá trị bất thường: {numeric_cols}")

# Phát hiện giá trị bất thường
outliers_mask, outlier_info = detect_outliers_iqr(df_processed, numeric_cols, multiplier=1.5)

print(f"\nTổng giá trị bất thường phát hiện: {outliers_mask.sum()} ({outliers_mask.sum()/len(df_processed)*100:.2f}%)")
print("\nChi tiết giá trị bất thường:")
for col, info in outlier_info.items():
    print(f"{col}: {info['count']} bất thường (khoảng: [{info['lower_bound']:.3f}, {info['upper_bound']:.3f}])")

# Vẽ biểu đồ hộp cho các cột chính
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols[:6]):
    axes[idx].boxplot(df_processed[col])
    axes[idx].set_title(f'Biểu đồ hộp - {col}')
    axes[idx].set_ylabel(col)

plt.tight_layout()
plt.show()

print(f"\nHình dạng dữ liệu trước khi loại bỏ bất thường: {df_processed.shape}")


## 5. Chuẩn Hóa/Scaling (Tùy Chọn)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Xác định các biến số (loại trừ biến mục tiêu)
feature_cols = [col for col in df_processed.columns if col != 'Rings']
numeric_feature_cols = df_processed[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

print(f"Các biến số để chuẩn hóa: {numeric_feature_cols}")

# Khởi tạo và huấn luyện bộ chuẩn hóa
scaler = StandardScaler()
df_scaled = df_processed.copy()
df_scaled[numeric_feature_cols] = scaler.fit_transform(df_processed[numeric_feature_cols])

print("\nDữ liệu sau chuẩn hóa (5 hàng đầu):")
print(df_scaled.head())
print("\nThống kê sau chuẩn hóa:")
print(df_scaled[numeric_feature_cols].describe())

# So sánh trước và sau chuẩn hóa
print("\n--- Trước chuẩn hóa ---")
print(df_processed[numeric_feature_cols].describe().loc[['mean', 'std']])
print("\n--- Sau chuẩn hóa ---")
print(df_scaled[numeric_feature_cols].describe().loc[['mean', 'std']])


## 6. Tóm Tắt Cuối Cùng & Lưu Lại

In [ ]:
print("=" * 60)
print("TIỀN XỬ LỰ DỮ LIỆU - TÓM TẮT")
print("=" * 60)

print(f"\n1. DỮ LIỆU GỐC:")
print(f"   - Hình dạng: {df.shape}")
print(f"   - Giá trị thiếu: {df.isnull().sum().sum()}")
print(f"   - Dòng trùng lặp: {df.duplicated().sum()}")

print(f"\n2. SAU MÃ HÓA:")
print(f"   - Hình dạng: {df_processed.shape}")
print(f"   - Các cột: {df_processed.columns.tolist()}")

print(f"\n3. PHÁT HIỆN GIÁC TRỊ BẤT THƯỜNG (Phương pháp IQR):")
print(f"   - Tổng bất thường tìm thấy: {outliers_mask.sum()}")
print(f"   - Tỷ lệ: {outliers_mask.sum()/len(df_processed)*100:.2f}%")

print(f"\n4. CHUẨN HÓA:")
print(f"   - Phương pháp: StandardScaler")
print(f"   - Số cơ bản được chuẩn hóa: {len(numeric_feature_cols)}")

print(f"\n5. DỮ LIỆU CUỐI CÙNG:")
print(f"   - Hình dạng: {df_scaled.shape}")
print(f"   - Sẵn sàng cho mô hình: CÓ")

# Lưu những tập dữ liệu đã xử lý
processed_path = Path.cwd().parent.parent / 'data' / 'interim'
processed_path.mkdir(parents=True, exist_ok=True)

# Lưu không chuẩn hóa
df_processed.to_csv(processed_path / 'abalone_processed_no_scaling.csv', index=False)
print(f"\n✓ Đã lưu (không chuẩn hóa): {processed_path / 'abalone_processed_no_scaling.csv'}")

# Lưu với chuẩn hóa
df_scaled.to_csv(processed_path / 'abalone_processed_scaled.csv', index=False)
print(f"✓ Đã lưu (chuẩn hóa): {processed_path / 'abalone_processed_scaled.csv'}")

print("\n" + "=" * 60)
print("TIỀN XỬ LỰ HOÀN THÀNH!")
print("=" * 60)
